# Day 9 — Joins

**What you will learn:**
- All 7 join types: inner, left, right, full, cross, semi, anti
- Joining on multiple columns
- Resolving column name conflicts after joins
- Broadcast joins — when and how
- Common pitfalls that appear on the exam

**Exam relevance:** DataFrame API (30%) — joins are one of the heaviest-tested topics.

In [ ]:
from pyspark.sql.functions import col, broadcast

employees = spark.createDataFrame([
    (1, "Alice",  10, "RO"),
    (2, "Bob",    20, "RO"),
    (3, "Carol",  10, "UK"),
    (4, "Dave",   30, "DE"),
    (5, "Eve",    99, "RO"),   # dept 99 — no match in departments
], ["emp_id", "name", "dept_id", "country"])

departments = spark.createDataFrame([
    (10, "Engineering", True),
    (20, "Marketing",   True),
    (30, "Sales",       False),
    (40, "Finance",     True),  # dept with no employees
], ["dept_id", "dept_name", "hiring"])

employees.show()
departments.show()

## 1. The 7 Join Types

| Join | Rows kept |
|---|---|
| `inner` | Only rows with a match in BOTH DataFrames |
| `left` | All rows from left + matching from right (null if no match) |
| `right` | All rows from right + matching from left (null if no match) |
| `full` / `outer` | All rows from both, null where no match |
| `cross` | Every row × every row (cartesian product) |
| `left_semi` | Only left rows that HAVE a match in right (no right columns) |
| `left_anti` | Only left rows that DO NOT have a match in right |

In [ ]:
# INNER — only matched rows (Eve and Finance dept are excluded)
print("=== INNER ===")
employees.join(departments, on="dept_id", how="inner").show()

# LEFT — all employees, Finance dept excluded
print("=== LEFT ===")
employees.join(departments, on="dept_id", how="left").show()

# FULL OUTER — all employees AND all departments
print("=== FULL OUTER ===")
employees.join(departments, on="dept_id", how="full").show()

In [ ]:
# SEMI — employees who have a valid dept (no right-side columns returned)
print("=== LEFT SEMI ===")
employees.join(departments, on="dept_id", how="left_semi").show()

# ANTI — employees with NO matching dept (Eve only)
print("=== LEFT ANTI ===")
employees.join(departments, on="dept_id", how="left_anti").show()

## 2. Handling Column Name Conflicts

When both DataFrames have a column with the same name (other than the join key), Spark keeps both — but you get ambiguous column references.

> **Exam tip:** Always alias DataFrames or rename conflicting columns BEFORE joining.

In [ ]:
# Both DataFrames have dept_id — joining on a string avoids duplication
# But if BOTH had a column called 'name', you'd get ambiguity:
emp2 = spark.createDataFrame([(1, "Alice", 10), (2, "Bob", 20)], ["id", "name", "dept_id"])
dept2 = spark.createDataFrame([(10, "Engineering"), (20, "Marketing")], ["dept_id", "name"])

# Alias the DataFrames to disambiguate
result = emp2.alias("e").join(dept2.alias("d"), on="dept_id") \
             .select(col("e.name").alias("emp_name"), col("d.name").alias("dept_name"))
result.show()

## 3. Joining on Multiple Columns

In [ ]:
orders = spark.createDataFrame([
    (1, "RO", 500), (2, "RO", 300), (3, "UK", 700), (4, "DE", 200)
], ["emp_id", "country", "amount"])

# Join on both emp_id AND country — both must match
employees.join(orders, on=["emp_id", "country"], how="inner").show()

## 4. Broadcast Joins

When one DataFrame is small enough to fit in memory, broadcasting it to all executors avoids a full shuffle.

| | Regular Join | Broadcast Join |
|---|---|---|
| Shuffle | Both sides shuffled | Only large side shuffled |
| When to use | Both sides large | One side small (< few hundred MB) |
| How | Default | `broadcast(small_df)` hint |

> **Exam tip:** AQE can switch to broadcast automatically if the small side is detected at runtime. You can also force it with `broadcast()` or set `spark.sql.autoBroadcastJoinThreshold` (default 10MB).

In [ ]:
# Explicitly broadcast the small departments table
result = employees.join(broadcast(departments), on="dept_id", how="inner")
result.show()

# Check explain to confirm BroadcastHashJoin in the physical plan
result.explain()

# Check/set broadcast threshold
print("Broadcast threshold:", spark.conf.get("spark.sql.autoBroadcastJoinThreshold"))

## 5. Common Exam Pitfalls

| Pitfall | What happens | Fix |
|---|---|---|
| Joining on column object from wrong DF | `AnalysisException` | Use string column name or alias |
| Ambiguous column after join | `AnalysisException` on select | Alias DFs before join |
| Cross join without condition | Cartesian product, OOM | Always specify join condition |
| `left_semi` expecting right columns | They're excluded | Use `inner` if you need right columns |
| Null keys | Nulls never match | Filter nulls before joining |

In [ ]:
# Nulls do NOT match in joins — always check
null_df = spark.createDataFrame([(1, "Alice", None), (2, "Bob", 10)], ["id", "name", "dept_id"])
print("Rows with null dept_id after inner join:")
null_df.join(departments, on="dept_id", how="inner").show()  # null row excluded

# Cross join — explicit keyword required (safety guard)
small_a = spark.createDataFrame([(1,), (2,)], ["a"])
small_b = spark.createDataFrame([("X",), ("Y",)], ["b"])
small_a.crossJoin(small_b).show()  # 2 × 2 = 4 rows

---

## Day 9 Checklist

- [ ] Know all 7 join types and which rows each keeps
- [ ] Know that `left_semi` returns only left-side columns
- [ ] Know that `left_anti` returns rows with NO match
- [ ] Resolved column name conflicts using DataFrame aliases
- [ ] Used `broadcast()` and verified in `explain()` output
- [ ] Know that null keys never match in any join type
- [ ] Know the default broadcast threshold (10MB) and how AQE overrides it

**Next:** Day 10 — String & Date Functions